# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

## 2. Datos

In [ ]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [ ]:
df.info()


In [ ]:
df.describe()


### 2.2 Definir X e y


In [ ]:
# Feature engineering básico: convertimos Ram y Weight (texto con unidades) a numéricas.
# Para esta primera versión usamos solo columnas simples; el resto (Cpu, Gpu, Memory,
# ScreenResolution, Product) se quedan fuera por ahora para futuras iteraciones.

def clean_features(data):
    data = data.copy()
    data['Ram_GB'] = data['Ram'].str.replace('GB', '', regex=False).astype(int)
    data['Weight_kg'] = data['Weight'].str.replace('kg', '', regex=False).astype(float)
    return data

df = clean_features(df)

feature_cols = ['Company', 'TypeName', 'Inches', 'Ram_GB', 'Weight_kg', 'OpSys']
categorical_cols = ['Company', 'TypeName', 'OpSys']
numeric_cols = ['Inches', 'Ram_GB', 'Weight_kg']

X = df[feature_cols]
y = df['Price_in_euros']


### 2.3 Dividir en train y test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [ ]:
# One-hot encoding de las categóricas. Random Forest no necesita escalado de numéricas,
# así que para esta v1 no usamos ningún scaler.
X_train_enc = pd.get_dummies(X_train, columns=categorical_cols)
X_test_enc = pd.get_dummies(X_test, columns=categorical_cols)

# Alineamos columnas por si test tiene categorías que no aparecen en train (o viceversa)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)


## 4. Modelado

### 4.1 Entrenamiento

In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train_enc, y_train)


### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [ ]:
y_pred = model.predict(X_test_enc)
rmse = root_mean_squared_error(y_test, y_pred)
print(f'RMSE: {rmse:.2f}')


### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
# v1: sin optimizar hiperparámetros todavía. Pendiente para próximas submissions
# (GridSearchCV / RandomizedSearchCV, probar SVR, añadir más features...).


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [ ]:
X_enc = pd.get_dummies(X, columns=categorical_cols)

final_model = RandomForestRegressor(random_state=42)
final_model.fit(X_enc, y)


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [ ]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [ ]:
X_pred = clean_features(X_pred)
X_pred_feat = X_pred[feature_cols]

X_pred_enc = pd.get_dummies(X_pred_feat, columns=categorical_cols)
X_pred_enc = X_pred_enc.reindex(columns=X_enc.columns, fill_value=0)


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [ ]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

### 8.2 Crea tu submission

In [ ]:
predictions = final_model.predict(X_pred_enc)

submission = pd.DataFrame({
    'laptop_ID': X_pred['laptop_ID'],
    'Price_in_euros': predictions
})
submission.head()


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [ ]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [ ]:
checker(submission, sample)